# Training: EOG→EEG Linear Regression Baseline

**Purpose:** 

Fit a linear regression from peripheral channels (hEOG, vEOG) to the
target EEG channel (Fz, Pz, Oz), under the SAME GroupKFold splits as
the video model. Provides a reference point for how much of the
reported performance could be explained by ocular activity.

**Scope:**

- **Subjects:** VALID_SUBJECTS, [1, 2, 4, 6, 7, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22]
- **Channels:** Fz, Pz, Oz
- **References:** hEOG, vEOG
- **CV:** GroupKFold, 5 folds over 40 trials
- **Preprocessing:** 4–45 Hz band-pass on EOG, per-fold z-scoring on train

**Output directory:** `EOG-Regression/Results/`

**Note:** No GPU needed. No checkpoint saving. The pickle is written
directly to `save_dir`.

In [ ]:
# ============================================================
# 0. Environment setup (Colab or local)
# ============================================================
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ============================================================
# 1. Standard library
# ============================================================
import os
import sys
import traceback

# ============================================================
# 2. Project paths
# ============================================================
if IN_COLAB:
    SOURCE_ROOT = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Source'
    DATA_ROOT   = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Data'
else:
    SOURCE_ROOT = r'D:\Thesis\Codes\Source'
    DATA_ROOT   = r'D:\Thesis\Codes\Data'

if not os.path.isdir(os.path.join(SOURCE_ROOT, 'face2eeg')):
    raise FileNotFoundError(
        f"face2eeg package not found under SOURCE_ROOT = {SOURCE_ROOT}. "
        f"Expected directory: {os.path.join(SOURCE_ROOT, 'face2eeg')}"
    )

if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)


# ============================================================
# 3. Package imports
# ============================================================
from face2eeg.core.experiments import run_eog_regression
from face2eeg.core.config      import SEED, FS_EEG, VALID_SUBJECTS


In [ ]:
# ============================================================
# Configuration
# ============================================================
BASE = os.path.join(DATA_ROOT, 'Experiments', 'Cross-Validation')

EXP_DIR     = os.path.join(BASE, 'EOG-Regression')
RESULTS_DIR = os.path.join(EXP_DIR, 'Results')

os.makedirs(RESULTS_DIR, exist_ok=True)


# ============================================================
# Experiment scope
# ============================================================
SUBJECTS = VALID_SUBJECTS
CHANNELS = ['Fz', 'Pz', 'Oz']


# ============================================================
# Regression settings
# ============================================================
N_POINTS   = 40
N_SPLITS   = 5
REFERENCES = ('heog', 'veog')
JSON_NAME  = 'EOG2EEG_summary.json'


# ============================================================
# Sanity check
# ============================================================
print("EXP_DIR     :", EXP_DIR)
print("RESULTS_DIR :", RESULTS_DIR)
print()
print("SUBJECTS    :", SUBJECTS)
print("CHANNELS    :", CHANNELS)
print()
print("N_POINTS    :", N_POINTS)
print("N_SPLITS    :", N_SPLITS)
print("REFERENCES  :", REFERENCES)
print()
print("FS_EEG      :", FS_EEG, " (from config)")

## Run: EOG regression per (subject, channel)

For each subject and each channel, fit `n_splits` per-fold linear
regressions from `references` (hEOG + vEOG) to the target EEG
channel. Results are saved as `sub{XX}_ch{CH}.pkl` with the same
schema as `run_kfold_pipeline`.

In [ ]:
for subject_id in SUBJECTS:
    print(f"\n{'#'*70}")
    print(f"#  SUBJECT {subject_id:02d}")
    print(f"{'#'*70}")

    for channel in CHANNELS:
        print(f"\n{'='*60}")
        print(f"  Subject {subject_id} | Channel {channel}")
        print(f"{'='*60}")

        save_path_result = os.path.join(
            RESULTS_DIR, f'sub{subject_id:02d}_ch{channel}.pkl'
        )
        if os.path.isfile(save_path_result):
            print(f"  [skip] already exists: {save_path_result}")
            continue

        try:
            run_eog_regression(
                subject_id=subject_id,
                channel_name=channel,
                n_points=N_POINTS,
                n_splits=N_SPLITS,
                references=REFERENCES,
                fs=FS_EEG,
                save_dir=RESULTS_DIR,
                json_filename=JSON_NAME,
                root_dir=DATA_ROOT,
                verbose=True,
            )
        except Exception as e:
            print(f"  !! ERROR for subject={subject_id}, channel={channel}: {e}")
            traceback.print_exc()
            continue

In [ ]:
print("\n" + "="*70)
print("DONE — EOG Regression")
print("="*70)
print(f"Results dir: {RESULTS_DIR}")
print()

for sid in SUBJECTS:
    for ch in CHANNELS:
        p = os.path.join(RESULTS_DIR, f'sub{sid:02d}_ch{ch}.pkl')
        ok = "OK" if os.path.isfile(p) else "--"
        print(f"  [{ok}]  S{sid:02d} | {ch}  {os.path.basename(p)}")